In [490]:
import os
import pandas as pd
from datetime import datetime
import importlib
import numpy as np

np.set_printoptions(suppress=True)

In [491]:
# Relative imports
os.chdir("..")
import hidden_state_model.processor
importlib.reload(hidden_state_model.processor)
Processor = hidden_state_model.processor.Processor
os.chdir("hidden_state_model")

### Read (and compact) dataframes

In [492]:
compact = False

In [493]:
# Iterate over files in dfs/*.parquet and combine to one df
dfs = []

read = []
for file in os.listdir("data"):
    if file.endswith(".parquet"):
        read.append(file)
        print(f"Reading {file}")
        df = pd.read_parquet(f"data/{file}")
        dfs.append(df)
    if file.endswith(".csv"):
        read.append(file)
        df = pd.read_csv(f"data/{file}", index_col=0)
        dfs.append(df)

raw_df = pd.concat(dfs)

if compact and len(dfs) > 1:
    print("Compacintg dfs")
    # Move read files to trash and write combined df to dfs/combined_{timestamp}.parquet
    trash = "data/trash"
    for f in read:
        os.rename(f"data/{f}", f"{trash}/{f}")

    timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
    raw_df.to_parquet(f"data/compacted_{timestamp}.parquet")

dfs = []  # Clear memory
raw_df

Reading fixed.parquet


,prev_entry,public_cards,player_piles,current_player_i,bet_in_stage,bet_in_game,player_has_played,player_is_folded,first_better_i,big_blind,player_name,player_type,action,amount,p,relative_ev,rank,tiebreakers
state_id,,,,,,,,,,,,,,,,,,
4896996752,<NA>,[],"[98, 96]",0,"[2, 4]","[2, 4]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,call,2,0.5130,0.015390,0,"[8, 4, 0, 0, 0]"
6158719728,4896996752,"[41, 30, 42]","[96, 96]",0,"[0, 0]","[4, 4]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,raise,4,0.6077,0.024308,1,"[4, 8, 3, 2, 0]"
6158687296,6158719728,"[41, 30, 42, 6]","[92, 92]",0,"[0, 0]","[8, 8]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,raise,8,0.5770,0.046160,1,"[4, 8, 6, 3, 0]"
6158493008,6158687296,"[41, 30, 42, 6, 28]","[84, 84]",0,"[0, 0]","[16, 16]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,check,0,0.5421,0.086736,2,"[4, 2, 8, 0, 0]"
6158678432,6158493008,"[41, 30, 42, 6, 28]","[84, 78]",0,"[0, 6]","[16, 22]","[True, True]","[False, False]",0,4,Tord,HumanPlayer,call,6,0.5421,0.102999,2,"[4, 2, 8, 0, 0]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6431248928,6127529072,"[8, 9, 36]","[152, 40]",0,"[0, 0]","[4, 4]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,raise,4,0.6240,0.024960,1,"[8, 10, 9, 4, 0]"
6431104400,6431248928,"[8, 9, 36, 6]","[148, 36]",0,"[0, 0]","[8, 8]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,raise,4,0.5459,0.043672,1,"[8, 10, 9, 6, 0]"
6127537840,6431104400,"[8, 9, 36, 6]","[144, 26]",0,"[4, 10]","[12, 18]","[True, True]","[False, False]",0,4,Tord,HumanPlayer,call,6,0.5459,0.081885,1,"[8, 10, 9, 6, 0]"


In [494]:
raw_df.dtypes

prev_entry            object
public_cards          object
player_piles          object
current_player_i       int64
bet_in_stage          object
bet_in_game           object
player_has_played     object
player_is_folded      object
first_better_i         int64
big_blind              int64
player_name           object
player_type           object
action                object
amount                 int64
p                    float64
relative_ev          float64
rank                   int64
tiebreakers           object
dtype: object

In [495]:
# Check for conflicting rows
dupe_df = raw_df[raw_df.index.duplicated()]
assert len(dupe_df) == 0, dupe_df

## Process data

In [496]:
processor = Processor(raw_df)
df = processor.get_processed_df()
df

,game_id,raise_preflop,raise_flop,raise_turn,raise_river,raise_showdown,call_preflop,call_flop,call_turn,call_river,...,opponent_check_turn,opponent_check_river,opponent_check_showdown,action,amount,excess_rank,p,relative_ev,stage,player_name
4896996752,d1f89c2a-87c8-4200-8c9b-8600669db907,0,0,0,0,0,0,0,0,0,...,0,0,0,call,2,0,0.5130,0.015390,preflop,Tord
6158719728,d1f89c2a-87c8-4200-8c9b-8600669db907,0,0,0,0,0,2,0,0,0,...,0,0,0,raise,4,1,0.6077,0.024308,flop,Tord
6158687296,d1f89c2a-87c8-4200-8c9b-8600669db907,0,4,0,0,0,2,0,0,0,...,0,0,0,raise,8,1,0.5770,0.046160,turn,Tord
6158493008,d1f89c2a-87c8-4200-8c9b-8600669db907,0,4,8,0,0,2,0,0,0,...,0,0,0,check,0,1,0.5421,0.086736,river,Tord
6158678432,d1f89c2a-87c8-4200-8c9b-8600669db907,0,4,8,0,0,2,0,0,0,...,0,0,0,call,6,1,0.5421,0.102999,river,Tord
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6431248928,2241c902-053e-4594-aaf4-521885a81585,0,0,0,0,0,2,0,0,0,...,0,0,0,raise,4,1,0.6240,0.024960,flop,Tord
6431104400,2241c902-053e-4594-aaf4-521885a81585,0,4,0,0,0,2,0,0,0,...,0,0,0,raise,4,1,0.5459,0.043672,turn,Tord
6127537840,2241c902-053e-4594-aaf4-521885a81585,0,4,4,0,0,2,0,0,0,...,0,0,0,call,6,1,0.5459,0.081885,turn,Tord
6431250896,2241c902-053e-4594-aaf4-521885a81585,0,4,4,0,0,2,0,6,0,...,0,1,0,check,0,0,0.9412,0.169416,river,Tord


In [497]:
df.dtypes

game_id                     object
raise_preflop                int64
raise_flop                   int64
raise_turn                   int64
raise_river                  int64
raise_showdown               int64
call_preflop                 int64
call_flop                    int64
call_turn                    int64
call_river                   int64
call_showdown                int64
check_preflop                int64
check_flop                   int64
check_turn                   int64
check_river                  int64
check_showdown               int64
opponent_raise_preflop       int64
opponent_raise_flop          int64
opponent_raise_turn          int64
opponent_raise_river         int64
opponent_raise_showdown      int64
opponent_call_preflop        int64
opponent_call_flop           int64
opponent_call_turn           int64
opponent_call_river          int64
opponent_call_showdown       int64
opponent_check_preflop       int64
opponent_check_flop          int64
opponent_check_turn 

## Training

In [498]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [499]:
X = df.drop(["excess_rank", "game_id", "p", "relative_ev"], axis=1)
y = df["excess_rank"]
groups = df["game_id"]  # Group by 'game_id' to ensure no data leakage

In [500]:
X

,raise_preflop,raise_flop,raise_turn,raise_river,raise_showdown,call_preflop,call_flop,call_turn,call_river,call_showdown,...,opponent_call_showdown,opponent_check_preflop,opponent_check_flop,opponent_check_turn,opponent_check_river,opponent_check_showdown,action,amount,stage,player_name
4896996752,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,call,2,preflop,Tord
6158719728,0,0,0,0,0,2,0,0,0,0,...,0,0,1,0,0,0,raise,4,flop,Tord
6158687296,0,4,0,0,0,2,0,0,0,0,...,0,0,1,0,0,0,raise,8,turn,Tord
6158493008,0,4,8,0,0,2,0,0,0,0,...,0,0,1,0,0,0,check,0,river,Tord
6158678432,0,4,8,0,0,2,0,0,0,0,...,0,0,1,0,0,0,call,6,river,Tord
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6431248928,0,0,0,0,0,2,0,0,0,0,...,0,0,1,0,0,0,raise,4,flop,Tord
6431104400,0,4,0,0,0,2,0,0,0,0,...,0,0,1,0,0,0,raise,4,turn,Tord
6127537840,0,4,4,0,0,2,0,0,0,0,...,0,0,1,0,0,0,call,6,turn,Tord
6431250896,0,4,4,0,0,2,0,6,0,0,...,0,0,1,0,1,0,check,0,river,Tord


In [501]:
y

4896996752    0
6158719728    1
6158687296    1
6158493008    1
6158678432    1
             ..
6431248928    1
6431104400    1
6127537840    1
6431250896    0
6431104448    0
Name: excess_rank, Length: 3741, dtype: int64

In [502]:
# Identify categorical columns (excluding 'game_id')
categorical_cols = ["action", "stage", "player_name"]

# Preprocessing pipeline: OneHotEncoding for categorical and scaling for numerical
preprocessor = ColumnTransformer(
    transformers=[("cat", OneHotEncoder(drop="first"), categorical_cols)],
    remainder="passthrough",
)

# Create the full pipeline with logistic regression
model = Pipeline(
    [
        ("preprocess", preprocessor),
        (
            "classifier",
            LogisticRegression(
                multi_class="multinomial", solver="lbfgs", max_iter=1000
            ),
        ),
    ]
)

In [503]:
# Grouped train-test split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")

Train shape: (2979, 34)
Test shape: (762, 34)


In [504]:
# Train the model
model.fit(X_train, y_train)

# Evaluate the model
accuracy = model.score(X_test, y_test)
print(f"Accuracy: {accuracy}")

Accuracy: 0.7545931758530183


In [505]:
probabilities = model.predict_proba(X_test)
prob_df = pd.DataFrame(probabilities, columns=model.classes_)
prob_df["true"] = y_test.values
prob_df["pred"] = model.predict(X_test)
prob_df["correct"] = prob_df["true"] == prob_df["pred"]
prob_df["goodness"] = prob_df.apply(lambda x: x.get(x["true"]) or 0, axis=1)
print("Accuracy", prob_df["correct"].mean())
print("Mean goodness: ", prob_df["goodness"].mean())
prob_df

Accuracy 0.7545931758530183
Mean goodness:  0.6728111620275163


,0,1,2,3,4,5,true,pred,correct,goodness
0,0.635234,0.332956,2.463084e-02,0.003857,0.001745,1.577815e-03,0,0,True,0.635234
1,0.564900,0.428571,5.633284e-07,0.004628,0.001898,2.316266e-06,1,0,False,0.428571
2,0.579608,0.414409,1.798899e-07,0.004046,0.001935,1.134361e-06,1,0,False,0.414409
3,0.462861,0.509076,1.286441e-06,0.006198,0.021850,1.310788e-05,1,1,True,0.509076
4,0.429265,0.559299,7.277520e-07,0.003314,0.008117,5.059659e-06,1,1,True,0.559299
...,...,...,...,...,...,...,...,...,...,...
757,0.645362,0.254112,2.948314e-03,0.023990,0.022906,5.068119e-02,0,0,True,0.645362
758,0.966660,0.032680,7.741071e-05,0.000119,0.000182,2.813559e-04,0,0,True,0.966660
759,0.131213,0.757074,6.312785e-02,0.015780,0.013386,1.941947e-02,1,1,True,0.757074
760,0.731543,0.251054,7.523233e-03,0.002987,0.003160,3.732966e-03,0,0,True,0.731543


In [506]:
# Look at incorrect predictions
print(prob_df[prob_df["correct"] == False].shape[0], "incorrect predictions:")
prob_df[prob_df["correct"] == False]

187 incorrect predictions:


,0,1,2,3,4,5,true,pred,correct,goodness
1,0.564900,0.428571,5.633284e-07,0.004628,1.898412e-03,2.316266e-06,1,0,False,0.428571
2,0.579608,0.414409,1.798899e-07,0.004046,1.935026e-03,1.134361e-06,1,0,False,0.414409
21,0.757052,0.214061,9.680247e-03,0.008498,2.291458e-03,8.416631e-03,1,0,False,0.214061
55,0.670700,0.229884,1.217218e-02,0.003168,5.758139e-02,2.649407e-02,1,0,False,0.229884
56,0.679966,0.276092,7.527409e-03,0.001851,2.338377e-02,1.117947e-02,1,0,False,0.276092
...,...,...,...,...,...,...,...,...,...,...
737,0.391771,0.560510,1.586819e-02,0.016185,8.030056e-03,7.635655e-03,0,1,False,0.391771
738,0.336843,0.530234,2.680939e-02,0.062145,2.340032e-02,2.056850e-02,0,1,False,0.336843
746,0.307332,0.562064,1.305216e-01,0.000082,1.417408e-07,2.142522e-10,0,1,False,0.307332
747,0.282739,0.607825,1.092968e-01,0.000138,1.604656e-07,4.744708e-10,0,1,False,0.282739
